# Toy master flat: HDRL's `hdrl_flat_compute`, ported to NumPy

This notebook makes fake flat-field readouts (no ScopeSim) and builds master flats
from them with the algorithm of HDRL's `hdrl_flat_compute`, to show how the method
behaves and what each of its inputs does.

- **How the original function works:** see [`README.md`](README.md) in this
  directory (a plain-English walk-through of `hdrl_flat.c`).
- **Source ported:** `~/Documents/git.repos/eso_cpl_hdrl/hdrl-1.6.0/hdrl_flat.c`
  (HDRL 1.6.0).
- **Interface copied:** [`flat.cpp`](flat.cpp), the PyHDRL binding:
  `Flat(filter_size_x, filter_size_y, method).compute(hdrl_data, collapse, stat_mask)`
  returning the named tuple `FlatResult(master, contrib_map)`.

**This is a port, not the real library.** PyHDRL isn't installed here, and CPL/HDRL
aren't built, so the steps of `hdrl_flat.c` are re-implemented in NumPy/SciPy. The
flat algorithm is followed line by line. Two pieces it calls are approximated:
- the median filter (`hdrl_parallel_filter_image`);
- the frame combination (`hdrl_imagelist_collapse`).

The last cell runs the real PyHDRL instead, if it is ever importable.

In [ ]:
from collections import namedtuple
from dataclasses import dataclass

import warnings

import matplotlib.pyplot as plt
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

plt.rcParams["figure.dpi"] = 70
plt.rcParams["image.origin"] = "lower"

## 1. Data model: an `hdrl_image` stand-in

An HDRL image carries, per pixel, a **value**, an **error** and a **bad-pixel flag**.
In CPL a mask pixel that is `True` means *rejected / bad*. The same convention
applies to `stat_mask` below: `True` = leave this pixel out (see `README.md`).

In [ ]:
@dataclass
class HdrlImage:
    # stand-in for hdrl_image (value, error, bad-pixel mask; True = bad, as in CPL)
    data: np.ndarray
    error: np.ndarray
    bpm: np.ndarray

    def copy(self):
        return HdrlImage(self.data.copy(), self.error.copy(), self.bpm.copy())

## 2. Masked median filter: `hdrl_parallel_filter_image(..., CPL_FILTER_MEDIAN)` stand-in

The flat algorithm smooths with a median filter over a box of
`filter_size_x` × `filter_size_y` pixels.
- **Bad pixels:** pixels flagged in the image's bad-pixel mask don't enter the median.
- **Edges:** near the image edges only the pixels inside the image are used (CPL's
  `CPL_BORDER_FILTER` behaviour).
- **No good pixels:** a pixel whose whole box is bad becomes bad.
- **Input bad pixels stay bad:** a pixel that is bad in the input stays bad in the
  output. `hdrl_flat.c` relies on this: it asserts that the two-region smoothing gives
  back the input bad-pixel mask.

See `README.md` for where the smoothing enters the two methods.

In [ ]:
def median_filter_masked(data, bpm, filter_size_x, filter_size_y, chunk_rows=32):
    # Stand-in for hdrl_parallel_filter_image(img, NULL, kernel, CPL_FILTER_MEDIAN)
    # (see README.md): median of the good pixels in a filter_size_y x filter_size_x box
    # around each pixel; returns (filtered, bpm_filtered).
    hy, hx = filter_size_y // 2, filter_size_x // 2
    work = np.where(bpm, np.nan, data.astype(float))
    padded = np.pad(work, ((hy, hy), (hx, hx)), constant_values=np.nan)
    windows = sliding_window_view(padded, (filter_size_y, filter_size_x))
    out = np.empty(data.shape)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN slices -> NaN
        for start in range(0, data.shape[0], chunk_rows):
            stop = min(start + chunk_rows, data.shape[0])
            out[start:stop] = np.nanmedian(windows[start:stop], axis=(2, 3))
    bad = ~np.isfinite(out) | bpm  # input bad pixels stay bad in the output
    return np.where(bad, 0.0, out), bad

## 3. Frame combination: `hdrl_imagelist_collapse` stand-in

The normalised frames are combined pixel by pixel. Bad pixels don't take part, and
the **contribution map** counts how many frames were used per pixel. The methods
mirror HDRL's collapse methods:
- `mean`, `weighted_mean`, `median`;
- `sigclip(kappa_low, kappa_high, niter)`;
- `minmax(nlow, nhigh)`.

The error formulas are close to, but not copied from, HDRL's `hdrl_collapse.c`:
- mean: √Σe² / n;
- weighted mean: 1 / √Σ(1/e²);
- median: √(π/2) × the mean's error (for n > 2);
- sigclip and minmax: the mean's error over the frames kept.

The sigma-clip here uses the median and the MAD as a robust centre and scale. See
`README.md`: in `hdrl_flat_compute` this step is simply "combine with the chosen
collapse method".

In [ ]:
class Collapse:
    # Stand-in for an HDRL collapse parameter (hdrl_collapse_*_parameter_create);
    # see README.md for its role in hdrl_flat_compute.
    def __init__(self, kind, **kwargs):
        if kind not in ("mean", "weighted_mean", "median", "sigclip", "minmax"):
            raise ValueError(f"unknown collapse kind {kind!r}")
        self.kind, self.kwargs = kind, kwargs

    def __repr__(self):
        args = ", ".join(f"{k}={v}" for k, v in self.kwargs.items())
        return f"Collapse({self.kind!r}{', ' if args else ''}{args})"

    def __call__(self, images):
        # returns (HdrlImage combined, contrib_map)
        data = np.array([im.data for im in images], dtype=float)
        err = np.array([im.error for im in images], dtype=float)
        good = ~np.array([im.bpm for im in images])

        if self.kind == "sigclip":
            k_lo = self.kwargs.get("kappa_low", 3.0)
            k_hi = self.kwargs.get("kappa_high", 3.0)
            for _ in range(self.kwargs.get("niter", 3)):
                vals = np.where(good, data, np.nan)
                with np.errstate(all="ignore"), warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN slices -> NaN
                    med = np.nanmedian(vals, axis=0)
                    mad = 1.4826 * np.nanmedian(np.abs(vals - med), axis=0)
                keep = (data >= med - k_lo * mad) & (data <= med + k_hi * mad)
                good &= keep | (mad == 0)
        elif self.kind == "minmax":
            n_lo, n_hi = self.kwargs.get("nlow", 1), self.kwargs.get("nhigh", 1)
            ranks = np.argsort(np.argsort(np.where(good, data, np.inf), axis=0), axis=0)
            n_good = good.sum(axis=0)
            good &= (ranks >= n_lo) & (ranks < n_good - n_hi)

        n = good.sum(axis=0)
        vals = np.where(good, data, np.nan)
        e2 = np.where(good, err**2, 0.0)
        with np.errstate(all="ignore"), warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN slices -> NaN
            if self.kind == "weighted_mean":
                w = np.where(good, 1.0 / err**2, 0.0)
                out = np.nansum(vals * w, axis=0) / w.sum(axis=0)
                out_err = 1.0 / np.sqrt(w.sum(axis=0))
            elif self.kind == "median":
                out = np.nanmedian(vals, axis=0)
                out_err = np.sqrt(e2.sum(axis=0)) / n
                out_err = np.where(n > 2, np.sqrt(np.pi / 2) * out_err, out_err)
            else:  # mean, sigclip, minmax: mean of the frames kept
                out = np.nanmean(vals, axis=0)
                out_err = np.sqrt(e2.sum(axis=0)) / n
        bad = n == 0
        combined = HdrlImage(
            np.where(bad, 0.0, out), np.where(bad, 0.0, out_err), bad
        )
        return combined, n.astype(int)

## 4. `hdrl_flat_compute`, line by line

A port of `hdrl_flat_compute` in `hdrl_flat.c`. The comments name the C steps they
mirror; see **`README.md`** for the explanation of what the original function does
and why.

The one deliberate difference: the C function **overwrites its input image list**
to save memory. This port works on copies, so your frames survive the call.

Two CPL behaviours are reproduced along the way:
- **Division by zero:** dividing by a zero pixel flags that pixel as bad
  (`cpl_image_divide`).
- **Bad pixels:** flagged pixels are excluded from every median.

In [ ]:
FlatResult = namedtuple("FlatResult", "master contrib_map")  # as in flat.cpp


def verify_flat_parameters(filter_size_x, filter_size_y, method):
    # hdrl_flat_parameter_verify (see README.md, "What goes in")
    if method not in ("low", "high"):
        raise ValueError("Unsupported method (HDRL_FLAT_FREQ_LOW / _HIGH)")
    for name, size in (("filter_size_x", filter_size_x),
                       ("filter_size_y", filter_size_y)):
        if size <= 0:
            raise ValueError(f"{name} must be > 0")
        if size % 2 != 1:
            raise ValueError(f"{name} must an odd number")  # (sic, HDRL's message)


def _divide(image, divisor, divisor_bpm=None):
    # cpl_image_divide on value and error: x / 0 -> bad pixel
    bad = image.bpm | (divisor == 0)
    if divisor_bpm is not None:
        bad |= divisor_bpm
    with np.errstate(all="ignore"):
        image.data = np.where(bad, 0.0, image.data / divisor)
        image.error = np.where(bad, 0.0, image.error / divisor)
    image.bpm = bad


def hdrl_flat_compute(hdrl_data, stat_mask, collapse_params,
                      filter_size_x, filter_size_y, method):
    # Port of hdrl_flat_compute (hdrl_flat.c, HDRL 1.6.0); see README.md for how
    # the original works. Returns FlatResult(master, contrib_map).
    #
    # hdrl_data       : list of HdrlImage, the input flats (copied, not overwritten)
    # stat_mask       : bool array or None; True = pixel left out of the statistics
    #                   (LOW) / the region smoothed separately (HIGH)
    # collapse_params : Collapse, how to combine the normalised frames
    # filter_size_x/y : odd, > 0; median-filter box
    # method          : 'low' (HDRL_FLAT_FREQ_LOW) or 'high' (HDRL_FLAT_FREQ_HIGH)

    # "Check Entries"
    if hdrl_data is None or len(hdrl_data) == 0:
        raise ValueError("No flatfields found")
    if collapse_params is None:
        raise ValueError("No collapsing parameter")
    verify_flat_parameters(filter_size_x, filter_size_y, method)
    lowf = method == "low"

    images = [im.copy() for im in hdrl_data]  # C: works in place
    for img in images:
        img_mask_orig = img.bpm.copy()

        if lowf:
            # LOW: normalise by the median of the frame; with a stat_mask, only the
            # pixels it doesn't flag count. No error propagation from the median.
            stat_bad = img_mask_orig | stat_mask if stat_mask is not None else img_mask_orig
            median = np.median(img.data[~stat_bad])
            img.data = img.data / median
            img.error = img.error / median
        else:
            if stat_mask is not None:
                # HIGH with a stat_mask: smooth the two regions separately so that
                # they don't influence each other across the boundary.
                # A) the region the stat_mask doesn't flag (e.g. illuminated)
                filt1, bad1 = median_filter_masked(
                    img.data, img_mask_orig | stat_mask, filter_size_x, filter_size_y)
                bad1 |= stat_mask  # re-apply the static mask: keep a sharp cutoff
                # B) the region the stat_mask flags (e.g. not illuminated)
                filt2, bad2 = median_filter_masked(
                    img.data, img_mask_orig | ~stat_mask, filter_size_x, filter_size_y)
                bad2 |= ~stat_mask
                # C) join them: fill rejected with 0, add; bpm = AND of the masks
                img_filtered = np.where(bad1, 0.0, filt1) + np.where(bad2, 0.0, filt2)
                filtered_bpm = bad1 & bad2
                # "& of static make should be input bpm" (assert in the C code)
                assert np.array_equal(filtered_bpm, img_mask_orig)
            else:
                img_filtered, filtered_bpm = median_filter_masked(
                    img.data, img_mask_orig, filter_size_x, filter_size_y)
            # divide value and error by the (noiseless) smoothed frame
            _divide(img, img_filtered, filtered_bpm)

        # "make sure that the error is positive" (TODO in the C code)
        img.error = np.abs(img.error)

    # "Combining the normalized flatfields generating the master-flatfield"
    comb_img, comb_ctr = collapse_params(images)

    if lowf:
        # LOW: median-smooth the combined image; keep its (unsmoothed) error
        filtered, filtered_bpm = median_filter_masked(
            comb_img.data, comb_img.bpm, filter_size_x, filter_size_y)
        master = HdrlImage(filtered, comb_img.error, filtered_bpm)
    else:
        master = comb_img
    return FlatResult(master, comb_ctr)

## 5. The `Flat` class, as in `flat.cpp`

Same interface as the PyHDRL binding in `flat.cpp`:
- the constructor validates its parameters as `hdrl_flat_parameter_create` does;
- `compute()` returns `FlatResult(master, contrib_map)`;
- the getters read the parameters back.

See `README.md` for what each parameter does.

In [ ]:
class Flat:
    # Mirrors hdrl::func::Flat in flat.cpp (see README.md for the algorithm)
    def __init__(self, filter_size_x, filter_size_y, method):
        verify_flat_parameters(filter_size_x, filter_size_y, method)  # _create
        self._fx, self._fy, self._method = filter_size_x, filter_size_y, method

    def compute(self, hdrl_data, collapse, stat_mask):
        return hdrl_flat_compute(hdrl_data, stat_mask, collapse,
                                 self._fx, self._fy, self._method)

    def get_filter_size_x(self):
        return self._fx

    def get_filter_size_y(self):
        return self._fy

    def get_method(self):
        return self._method

## 6. Fake readouts

256 × 256 pixel flats, 9 frames, built from known truths so the master flats can be
checked against them:

| ingredient | truth / purpose |
|---|---|
| illumination: vignetting (radial falloff, ~25% at the corners) × a 5% tilt × a 3% ripple (80 px period) | the **low-frequency** truth |
| pixel response: 2% Gaussian pixel-to-pixel variation × a few small dust "donuts" (~7 px across, 5% deep) | the **high-frequency** truth |
| unilluminated 20 px border (dark-subtracted, so only read noise) | what `stat_mask` is for |
| lamp level varying ±5% between frames | why each frame is normalised |
| photon noise (gain 2 e⁻/ADU) and read noise (10 e⁻), per-pixel error | error propagation |
| fixed dead and hot pixels, flagged in every frame's bad-pixel mask | bad pixels are ignored |
| cosmic rays, a few per frame, **not** flagged | the collapse has to reject them |

In [ ]:
rng = np.random.default_rng(42)
NY = NX = 256
BORDER = 20
N_FRAMES = 9
LAMP_E = 20000.0  # e- per pixel at the illumination peak
GAIN = 2.0  # e- / ADU
READ_NOISE_E = 10.0

yy, xx = np.indices((NY, NX))
lit = np.zeros((NY, NX), bool)
lit[BORDER:-BORDER, BORDER:-BORDER] = True
stat_mask = ~lit  # True = leave out: the unilluminated border

# low-frequency truth: vignetting x tilt x a mid-scale ripple (80 px period)
r2 = ((xx - NX / 2) ** 2 + (yy - NY / 2) ** 2) / (NX / 2) ** 2
RIPPLE_PERIOD = 80
illum = (
    (1.0 - 0.125 * r2)
    * (1.0 + 0.05 * (xx - NX / 2) / NX)
    * (1.0 + 0.03 * np.sin(2 * np.pi * xx / RIPPLE_PERIOD))
)

# high-frequency truth: pixel-to-pixel response x dust donuts
pix = 1.0 + 0.02 * rng.standard_normal((NY, NX))
dust_centres = [(70, 80), (150, 190), (200, 60), (110, 130)]
for cy, cx in dust_centres:
    rr = np.hypot(yy - cy, xx - cx)
    pix *= 1.0 - 0.05 * np.exp(-0.5 * ((rr - 3.0) / 1.0) ** 2)

# fixed bad pixels (in every frame's bpm)
bpm_fixed = rng.random((NY, NX)) < 0.003
dead = bpm_fixed & (rng.random((NY, NX)) < 0.5)
hot = bpm_fixed & ~dead

frames = []
lamp_levels = LAMP_E * (1.0 + 0.05 * rng.uniform(-1, 1, N_FRAMES))
for k in range(N_FRAMES):
    expected_e = np.where(lit, lamp_levels[k] * illum * pix, 0.0)
    counts_e = rng.poisson(expected_e) + READ_NOISE_E * rng.standard_normal((NY, NX))
    counts_e[dead] = 0.0
    counts_e[hot] = 50.0 * LAMP_E
    # cosmic rays: 1-3 px hits, not flagged
    for _ in range(25):
        cy, cx = rng.integers(0, NY - 3), rng.integers(0, NX - 3)
        h, w = rng.integers(1, 4, size=2)
        counts_e[cy:cy + h, cx:cx + w] += rng.uniform(5e3, 5e4)
    data = counts_e / GAIN
    error = np.sqrt(np.clip(counts_e, 0, None) + READ_NOISE_E**2) / GAIN
    frames.append(HdrlImage(data, error, bpm_fixed.copy()))

print(f"{N_FRAMES} frames, lamp levels {np.round(lamp_levels / LAMP_E, 3)} x LAMP_E")
print(f"bad pixels flagged: {bpm_fixed.sum()}, lit fraction {lit.mean():.2f}")

In [ ]:
def show(ax, img, title, mask=None, vmin=None, vmax=None, cmap="viridis"):
    img = np.where(mask, np.nan, img) if mask is not None else img
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)


def rms_in(values, region):
    return float(np.std(values[region]))


fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
show(axes[0], frames[0].data, "raw frame 0 [ADU]", vmin=0, vmax=1.2 * LAMP_E / GAIN)
show(axes[1], illum, "truth: illumination (low freq.)")
show(axes[2], pix, "truth: pixel response (high freq.)", vmin=0.93, vmax=1.07)
plt.tight_layout()

Comparison region: the illuminated area shrunk by half a filter box, plus the
pixels flagged bad, are left out of the residual statistics. The edges are looked
at separately in 7c.

In [ ]:
def inner_region(half_box):
    region = np.zeros((NY, NX), bool)
    b = BORDER + half_box
    region[b:-b, b:-b] = True
    return region & ~bpm_fixed


# expected noise of a 9-frame combination, relative, at the illumination median
rel_noise = np.sqrt(LAMP_E * np.median(illum[lit])) / (LAMP_E * np.median(illum[lit]))
print(f"photon noise per frame ~{100 * rel_noise:.2f}%, "
      f"over {N_FRAMES} frames ~{100 * rel_noise / np.sqrt(N_FRAMES):.2f}%")

## 7. Making flats

Every call below spells out all the inputs `hdrl_flat_compute` needs (see
`README.md`, "What goes in"): the frames, `stat_mask`, the collapse method,
`filter_size_x`, `filter_size_y` and `method`.

### 7a. Low-frequency flat (with `stat_mask`)

Each frame is divided by its median over the lit pixels, the frames are combined
with a sigma-clip, and the result is median-smoothed (`README.md`, "Low-frequency
flat").

In [ ]:
low = Flat(filter_size_x=15, filter_size_y=15, method="low")
res_low = low.compute(
    hdrl_data=frames,
    collapse=Collapse("sigclip", kappa_low=3.0, kappa_high=3.0, niter=3),
    stat_mask=stat_mask,
)
truth_low = illum / np.median((illum * pix)[lit & ~bpm_fixed])
region = inner_region(low.get_filter_size_x() // 2)
resid_low = res_low.master.data / truth_low - 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
show(axes[0], res_low.master.data, "LOW master flat", mask=~lit)
show(axes[1], resid_low, "LOW master / truth - 1", mask=~lit, vmin=-0.01, vmax=0.01,
     cmap="RdBu_r")
show(axes[2], res_low.contrib_map, "contribution map (frames used)", vmin=0,
     vmax=N_FRAMES)
plt.tight_layout()
print(f"LOW: rms(master/truth - 1) inside = {100 * rms_in(resid_low, region):.3f}%")

### 7b. High-frequency flat (with `stat_mask`)

Each frame is divided by its own median-smoothed copy, and the ratio frames are
combined (`README.md`, "High-frequency flat"). With a `stat_mask` the lit and dark
regions are smoothed separately.

The dark border of a HIGH master has no meaning (noise divided by noise), so it is
masked in the plots.

In [ ]:
high = Flat(filter_size_x=15, filter_size_y=15, method="high")
res_high = high.compute(
    hdrl_data=frames,
    collapse=Collapse("sigclip", kappa_low=3.0, kappa_high=3.0, niter=3),
    stat_mask=stat_mask,
)
region = inner_region(high.get_filter_size_x() // 2)
resid_high = res_high.master.data / pix - 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
show(axes[0], res_high.master.data, "HIGH master flat", mask=~lit | bpm_fixed,
     vmin=0.93, vmax=1.07)
show(axes[1], pix, "truth: pixel response", mask=~lit, vmin=0.93, vmax=1.07)
show(axes[2], resid_high, "HIGH master / truth - 1", mask=~lit | bpm_fixed,
     vmin=-0.02, vmax=0.02, cmap="RdBu_r")
plt.tight_layout()
print(f"HIGH: rms(master/truth - 1) inside = {100 * rms_in(resid_high, region):.3f}% "
      f"(the truth's own pixel-to-pixel rms is {100 * rms_in(pix - 1, region):.2f}%)")

### 7c. Why the `stat_mask`: the lit/dark boundary

Without a `stat_mask`, the smoothing box near the edge of the lit area also takes in
dark pixels. That pulls the smoothed copy down, so the ratio (frame / smoothed)
comes out too high in a band of about half a box width along the edge: **false
structure in the flat**.

With a `stat_mask`, the lit region is smoothed on its own. The band shrinks to a
small residual (~0.5% here): right at the edge the box only sees pixels on one side,
so it follows the illumination gradient less well.
The same applies to the LOW method's final smoothing, which doesn't use the
`stat_mask` at all: its edge band is shown too.

In [ ]:
res_high_nomask = high.compute(
    hdrl_data=frames,
    collapse=Collapse("sigclip", kappa_low=3.0, kappa_high=3.0, niter=3),
    stat_mask=None,
)

row = slice(BORDER + 40, NY - BORDER - 40)  # rows averaged for the profile
cols = np.arange(BORDER, BORDER + 40)  # lit columns from the left edge inwards


def edge_profile(img):
    vals = np.where(bpm_fixed, np.nan, img)[row][:, cols]
    return np.nanmean(vals, axis=0)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
zoom = (slice(BORDER - 5, BORDER + 40), slice(BORDER - 5, BORDER + 40))
show(axes[0], res_high.master.data[zoom], "HIGH, with stat_mask (corner zoom)",
     vmin=0.93, vmax=1.07)
show(axes[1], res_high_nomask.master.data[zoom], "HIGH, stat_mask=None (corner zoom)",
     vmin=0.93, vmax=1.07)
axes[2].plot(cols, edge_profile(res_high.master.data / pix),
             label="HIGH / truth, with stat_mask")
axes[2].plot(cols, edge_profile(res_high_nomask.master.data / pix),
             label="HIGH / truth, stat_mask=None")
axes[2].plot(cols, edge_profile(res_low.master.data / truth_low),
             label="LOW / truth (final smoothing ignores stat_mask)")
axes[2].axhline(1.0, color="k", lw=0.5)
axes[2].axvline(BORDER + high.get_filter_size_x() // 2, color="k", lw=0.5, ls="--")
axes[2].set_xlabel("column (lit area starts at %d; dashed: half a box in)" % BORDER)
axes[2].set_ylabel("mean over rows")
axes[2].set_title("profile across the left edge of the lit area", fontsize=9)
axes[2].legend(fontsize=8)
plt.tight_layout()

### 7d. The collapse method: cosmic rays

The cosmic rays are not in the bad-pixel masks, so only the collapse can remove
them. A plain `mean` keeps them; `sigclip` (or `minmax`, or `median`) rejects them.
Where pixels were rejected, the contribution map shows fewer than 9 frames.

In [ ]:
res_mean = high.compute(hdrl_data=frames, collapse=Collapse("mean"), stat_mask=stat_mask)
region = inner_region(high.get_filter_size_x() // 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
show(axes[0], res_mean.master.data, "HIGH, collapse = mean", mask=~lit | bpm_fixed,
     vmin=0.93, vmax=1.07)
show(axes[1], res_high.master.data, "HIGH, collapse = sigclip", mask=~lit | bpm_fixed,
     vmin=0.93, vmax=1.07)
show(axes[2], res_high.contrib_map, "sigclip contribution map", mask=~lit,
     vmin=N_FRAMES - 3, vmax=N_FRAMES)
plt.tight_layout()
for name, res in (("mean", res_mean), ("sigclip", res_high)):
    resid = res.master.data / pix - 1
    print(f"{name:8s}: rms {100 * rms_in(resid, region):.3f}%, "
          f"max |resid| {100 * np.max(np.abs(resid[region])):.1f}%")

### 7e. The filter box size

The box sets the line between "large scale" and "pixel scale" (`README.md`,
"Practical notes"):
- **too small (3 px):** the smoothed copy follows the dust donuts, so they drop out
  of the HIGH flat;
- **too large (61 px):** the smoothed copy no longer follows the 80 px illumination
  ripple, so the ripple leaks into the HIGH flat (stripes in its residual), where it
  doesn't belong.

The right box is larger than the pixel-scale structure (here the ~7 px dust donuts)
and clearly smaller than the illumination structure (here the 80 px ripple).

In [ ]:
sizes = (3, 15, 61)
fig, axes = plt.subplots(1, len(sizes) + 1, figsize=(18, 4.2))
cy, cx = dust_centres[1]
for ax, size in zip(axes, sizes):
    res = Flat(size, size, "high").compute(
        hdrl_data=frames,
        collapse=Collapse("sigclip", kappa_low=3.0, kappa_high=3.0, niter=3),
        stat_mask=stat_mask,
    )
    resid = res.master.data / pix - 1
    show(ax, resid, f"HIGH / truth - 1, filter {size}x{size}", mask=~lit | bpm_fixed,
         vmin=-0.03, vmax=0.03, cmap="RdBu_r")
    cut = np.where(bpm_fixed, np.nan, res.master.data)[cy, cx - 15:cx + 16]
    axes[-1].plot(np.arange(-15, 16), cut, label=f"filter {size}")
    print(f"filter {size:2d}: rms(master/truth - 1) = "
          f"{100 * rms_in(resid, inner_region(30)):.3f}%")
axes[-1].plot(np.arange(-15, 16), pix[cy, cx - 15:cx + 16], "k--", label="truth")
axes[-1].set_title("cut through a dust donut", fontsize=9)
axes[-1].set_xlabel("pixels from the donut centre")
axes[-1].legend(fontsize=8)
plt.tight_layout()

## 8. Full flat = LOW × HIGH, applied to a frame

The two flats are complementary: the LOW one holds the illumination shape, the
HIGH one the pixel-to-pixel response. Dividing a raw frame by their product (and by
its lamp level) should leave it flat to within the photon noise.

The exception is a rim along the edge of the lit area, about half a LOW box wide. It
comes from the LOW method's final smoothing: in `hdrl_flat.c` that step ignores the
`stat_mask`, so near the edge the box takes in dark pixels (see the green curve in
7c). In practice: trim that rim, or make the LOW flat on a region that extends beyond
the area you need.

In [ ]:
full_flat = res_low.master.data * res_high.master.data
region = inner_region(7)
raw = frames[0]
with np.errstate(all="ignore"):  # the flats are 0 at bad pixels
    corrected = raw.data / full_flat
corrected /= np.median(corrected[region])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
show(axes[0], full_flat, "full flat = LOW x HIGH", mask=~lit | bpm_fixed)
show(axes[1], raw.data / np.median(raw.data[region]), "raw frame 0 (normalised)",
     mask=~lit | bpm_fixed, vmin=0.7, vmax=1.1)
show(axes[2], corrected, "frame 0 / full flat (normalised)", mask=~lit | bpm_fixed,
     vmin=0.97, vmax=1.03)
plt.tight_layout()
clean = region & (np.abs(corrected - 1) < 0.1)  # leave out frame 0's cosmic rays
print(f"raw frame rms {100 * rms_in(raw.data / np.median(raw.data[region]) - 1, region):.2f}%"
      f" -> flat-fielded rms {100 * rms_in(corrected - 1, clean):.3f}% "
      f"(photon noise of one frame ~{100 * rel_noise:.2f}%)")

## 9. Optional: the real PyHDRL

If PyHDRL (the package `flat.cpp` belongs to) is importable, this cell runs the
same high-frequency case through the real `Flat` binding and compares the result
with the port. Building its inputs (`hdrl.core.ImageList`, `Collapse`, the CPL mask)
depends on the PyHDRL/PyCPL version, so check that part against your installation.

In [ ]:
try:
    import hdrl  # PyHDRL
except ImportError:
    print("PyHDRL is not installed: skipping the comparison with the real library.")
else:
    print("PyHDRL found:", hdrl.__file__)
    print("Build an hdrl.core.ImageList from `frames` (data, error, bpm), a collapse "
          "parameter and a cpl mask from `stat_mask`, then call\n"
          "  hdrl.func.Flat(11, 11, <HDRL_FLAT_FREQ_HIGH>).compute(imlist, collapse, mask)\n"
          "and compare its .master with res_high.master.data.")